In [ ]:
!wget https://jdbc.postgresql.org/download/postgresql-42.7.0.jar

--2026-05-02 18:00:59--  https://jdbc.postgresql.org/download/postgresql-42.7.0.jar
Resolving jdbc.postgresql.org (jdbc.postgresql.org)... 72.32.157.228, 2001:4800:3e1:1::228
Connecting to jdbc.postgresql.org (jdbc.postgresql.org)|72.32.157.228|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1077325 (1.0M) [application/java-archive]
Saving to: ‘postgresql-42.7.0.jar.9’

postgresql-42.7.0.j 100%[===================>]   1.03M   886KB/s    in 1.2s    

2026-05-02 18:01:02 (886 KB/s) - ‘postgresql-42.7.0.jar.9’ saved [1077325/1077325]



In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

work_dir = "/home/jovyan/work"
pg_jar      = f"{work_dir}/postgresql-42.7.0.jar"
delta_jar   = f"{work_dir}/delta-spark_2.12-3.2.0.jar"
storage_jar = f"{work_dir}/delta-storage-3.2.0.jar"
all_jars    = ",".join([pg_jar, delta_jar, storage_jar])

spark = (SparkSession.builder
         .appName("Gold-PreCheck")
         .config("spark.jars", all_jars)
         .config("spark.driver.extraClassPath", all_jars)
         .config("spark.executor.extraClassPath", all_jars)
         .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
         .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
         .config("spark.driver.memory", "4g")
         .getOrCreate()
)

SILVER = "/home/jovyan/work/data/silver"

fact = spark.read.format("delta").load(f"{SILVER}/fact_order_items")
print(f"Silver fact rows: {fact.count():,}")

orphan = fact.filter(F.col("department").isNull()).count()
print(f"Orphan rows (NULL department): {orphan:,}")
print(f"Orphan %: {orphan / fact.count() * 100:.4f}%")

Py4JJavaError: An error occurred while calling o42.load.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'spark_catalog': org.apache.spark.sql.delta.catalog.DeltaSparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1925)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.loadV2SessionCatalog(CatalogManager.scala:67)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$2(CatalogManager.scala:86)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$1(CatalogManager.scala:86)
	at scala.Option.map(Option.scala:230)
	at org.apache.spark.sql.connector.catalog.CatalogManager.v2SessionCatalog(CatalogManager.scala:85)
	at org.apache.spark.sql.connector.catalog.Catalog